In [1]:
# Install / upgrade dependencies
!pip install torch_geometric ogb --quiet

In [ ]:
import os
import csv
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
class AtomEncoder(nn.Module):
    VOCAB_SIZES = [128, 8, 16, 16, 16, 8, 16, 4, 4]

    def __init__(self, hidden_dim):
        super().__init__()
        self.embeddings = nn.ModuleList(
            [nn.Embedding(v, hidden_dim) for v in self.VOCAB_SIZES]
        )
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        # x: [B, N, 9]  integer features
        x = x.long()
        # Clamp each feature to its valid range to avoid index-out-of-bounds
        out = sum(
            emb(x[..., i].clamp(0, v - 1))
            for i, (emb, v) in enumerate(zip(self.embeddings, self.VOCAB_SIZES))
        )
        return self.proj(F.gelu(out))

In [7]:
class DenseGCNLayer(nn.Module):
    """
    Local branch: dense GCN with scalar edge gating.
    Identical to Peptides version.
    """
    def __init__(self, hidden_dim, edge_dim=3):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)   # scalar gate — avoids [B,N,N,H] blow-up
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        # x:               [B, N, H]
        # A_norm:          [B, N, N]
        # edge_attr_dense: [B, N, N, edge_dim]  or None
        if edge_attr_dense is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)  # [B,N,N]
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))

In [8]:
class SpectralMixMH(nn.Module):
    """
    Global branch: learned spectral filter over full Laplacian eigenbasis.
    Identical to Peptides version.
    """
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        # x: [B, N, H],  U: [B, N, N]
        x_hat      = torch.bmm(U.transpose(1, 2), x)         # spectral domain
        fil        = torch.sigmoid(self.filter_gen(x_hat))   # learned filter
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)                # back to node domain
        x_out      = x_out * mask.unsqueeze(-1)              # zero padded nodes
        return self.norm(self.out_proj(F.gelu(x_out)))

In [ ]:
class GatedPooling(nn.Module):
    """Gated graph-level readout. Identical to Peptides version."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x, mask):
        scores  = self.gate(x).squeeze(-1)                    # [B, N]
        scores  = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)  # [B, N, 1]
        return (x * weights).sum(dim=1)                       # [B, H]

In [11]:
def check_gate_health(model, val_loader):
    model.eval()
    batch = next(iter(val_loader)).to(device)

    with torch.no_grad():
        x, mask = to_dense_batch(batch.x.float(), batch.batch)
        adj     = to_dense_adj(batch.edge_index, batch.batch, max_num_nodes=x.size(1))
        I       = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj     = adj + I

        A_norm, U = model.compute_laplacian_basis(adj, mask)
        x_enc     = model.input_proj(x)
        k         = min(model.lap_k, U.size(-1))
        lap_pe    = U[:, :, :k] * mask.unsqueeze(-1)
        x_enc     = x_enc + model.pe_encoder(lap_pe)

        print("\n--- Gate Health Check ---")
        collapsed = False
        for i, layer in enumerate(model.layers):
            gate_vals = torch.sigmoid(layer["gate"](x_enc))
            mean_g    = gate_vals.mean().item()
            std_g     = gate_vals.std().item()

            if mean_g > 0.85:
                status    = "⚠️  COLLAPSED → GCN (spectral dead)"
                collapsed = True
            elif mean_g < 0.15:
                status    = "⚠️  COLLAPSED → SPECTRAL (GCN dead)"
                collapsed = True
            elif std_g < 0.05:
                status    = "⚠️  UNIFORM (not learning per-node routing)"
                collapsed = True
            else:
                status = "✅ HEALTHY"

            print(f"  Layer {i} | mean={mean_g:.4f} | std={std_g:.4f} | {status}")

        print("  ACTION: lr reset to 5e-4." if collapsed else "  All gates healthy.")
        print("-------------------------\n")

    model.train()
    return collapsed

In [2]:
!pip install rdkit

In [3]:
import torch

# Only patch if we haven't already!
if not hasattr(torch, "_is_patched"):
    _orig_load = torch.load
    def _patched_load(f, *args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _orig_load(f, *args, **kwargs)
    torch.load = _patched_load
    torch._is_patched = True

In [ ]:
# ==========================================
# GraphFNet — BBBP (Sanity Check)
# ==========================================
# !pip install torch_geometric scikit-learn --quiet

import os
import csv
import random
import time
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.datasets import MoleculeNet
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==========================================
# 1. Dataset Loader & Splitter
# ==========================================
class IndexedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        data = self.dataset[real_idx].clone()
        data.graph_idx = torch.tensor([real_idx], dtype=torch.long)
        return data

def load_bbbp(batch_size=32):
    dataset = MoleculeNet(root='./data', name='BBBP')

    # BBBP has ~2039 graphs. Let's do a simple 80/10/10 random split
    num_graphs = len(dataset)
    indices = list(range(num_graphs))
    random.shuffle(indices)

    train_split = int(0.8 * num_graphs)
    val_split   = int(0.9 * num_graphs)

    train_dataset = IndexedDataset(dataset, indices[:train_split])
    val_dataset   = IndexedDataset(dataset, indices[train_split:val_split])
    test_dataset  = IndexedDataset(dataset, indices[val_split:])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
    return train_loader, val_loader, test_loader, dataset


def precompute_spectral_cache(dataset, desc="Pre-computing spectral bases"):
    cache = [None] * len(dataset)
    for i in tqdm(range(len(dataset)), desc=desc):
        data = dataset[i]
        n   = data.num_nodes
        ei  = data.edge_index

        adj = torch.zeros(n, n)
        adj[ei[0], ei[1]] = 1.0
        adj.fill_diagonal_(1.0)

        deg          = adj.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt   = torch.diag(deg_inv_sqrt)
        A_norm       = D_inv_sqrt @ adj @ D_inv_sqrt

        L = torch.eye(n) - A_norm
        try:
            _, U = torch.linalg.eigh(L)
            max_abs_idx = torch.abs(U).argmax(dim=0)
            signs       = torch.sign(U[max_abs_idx, torch.arange(n)])
            signs[signs == 0] = 1.0
            U = U * signs.unsqueeze(0)
        except Exception:
            U = torch.eye(n)

        cache[i] = (A_norm, U)

    total_bytes = sum(a.numel() * 4 + u.numel() * 4 for a, u in cache)
    print(f"\nCache built: {len(cache)} graphs | Size: {total_bytes / 1024**2:.1f} MB")
    return cache


class AtomEncoder(nn.Module):
    VOCAB_SIZES = [128, 8, 16, 16, 16, 8, 16, 4, 4]
    def __init__(self, hidden_dim):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(v, hidden_dim) for v in self.VOCAB_SIZES])
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = x.long()
        out = sum(emb(x[..., i].clamp(0, v - 1)) for i, (emb, v) in enumerate(zip(self.embeddings, self.VOCAB_SIZES)))
        return self.proj(F.gelu(out))

class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=3):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))

class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))

class GatedPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(), nn.Linear(hidden_dim // 2, 1))
    def forward(self, x, mask):
        scores  = self.gate(x).squeeze(-1).masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)

class GraphFNet_BBBP(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, lap_k=8, dropout=0.1, edge_dim=3):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)
        self.input_proj = AtomEncoder(hidden_dim)
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "local":  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                "global": SpectralMixMH(hidden_dim, num_heads=num_heads),
                "gate":   nn.Linear(hidden_dim, hidden_dim),
                "norm":   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])

        self.pool = GatedPooling(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data, cache=None):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        B, N, _ = x.shape

        A_list, U_list = [], []
        for b, gidx in enumerate(data.graph_idx.tolist()):
            A_b, U_b = cache[gidx]
            n = A_b.size(0)
            A_list.append(F.pad(A_b, (0, N-n, 0, N-n)).to(x.device))
            U_list.append(F.pad(U_b, (0, N-n, 0, N-n)).to(x.device))
        A_norm = torch.stack(A_list)
        U      = torch.stack(U_list)

        if data.edge_attr is not None:
            edge_attr_dense = to_dense_adj(data.edge_index, data.batch, edge_attr=data.edge_attr[:, :3].float(), max_num_nodes=N)
        else:
            edge_attr_dense = None

        x      = self.input_proj(x)
        k      = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x      = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            x_res    = x
            x_local  = layer["local"](x, A_norm, edge_attr_dense)
            x_global = layer["global"](x, U, mask)
            gate     = torch.sigmoid(layer["gate"](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer["norm"](x_res + self.dropout(x_mix))

        x = x * mask.unsqueeze(-1)
        return self.classifier(self.pool(x, mask))


def evaluate_rocauc(model, loader, cache):
    model.eval()
    y_true_list, y_pred_list = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out   = model(batch, cache=cache)
            y_true_list.append(batch.y.cpu())
            y_pred_list.append(out.cpu())

    y_true = torch.cat(y_true_list, dim=0).numpy().flatten()
    y_pred = torch.cat(y_pred_list, dim=0).numpy().flatten()

    # Filter out NaNs (BBBP sometimes has missing labels depending on the PyG version)
    valid_idx = ~np.isnan(y_true)
    return roc_auc_score(y_true[valid_idx], y_pred[valid_idx])

def train_bbbp(max_epochs=100, patience=20, batch_size=32):
    # Set seed for reproducible split
    random.seed(42)
    train_loader, val_loader, test_loader, full_dataset = load_bbbp(batch_size)
    cache = precompute_spectral_cache(full_dataset)

    model = GraphFNet_BBBP().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_val, best_epoch, epochs_no_improve = 0.0, 0, 0
    ckpt_path = "best_graphfnet_bbbp.pt"

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch, cache=cache)
            valid_idx = ~torch.isnan(batch.y)

            loss = criterion(out[valid_idx], batch.y[valid_idx].float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        val_roc = evaluate_rocauc(model, val_loader, cache)
        improved = val_roc > best_val

        print(f"Epoch {epoch:03d} | Loss {total_loss/len(train_loader):.4f} | Val ROC-AUC {val_roc:.4f}" + (" ← best" if improved else ""))

        if improved:
            best_val, best_epoch, epochs_no_improve = val_roc, epoch, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(torch.load(ckpt_path, weights_only=False))
    test_roc = evaluate_rocauc(model, test_loader, cache)
    print(f"\nFINAL | Test ROC-AUC: {test_roc:.4f} | Best epoch: {best_epoch}")

if __name__ == "__main__":
    train_bbbp()

Device: cuda
Train: 1631 | Val: 204 | Test: 204


Pre-computing spectral bases: 100%|██████████| 2039/2039 [00:02<00:00, 820.14it/s]



Cache built: 2039 graphs | Size: 10.8 MB


Epoch 001 | Loss 0.5137 | Val ROC-AUC 0.7178 ← best


Epoch 002 | Loss 0.4345 | Val ROC-AUC 0.7368 ← best


...

Epoch 036 | Loss 0.0069 | Val ROC-AUC 0.8528


Epoch 037 | Loss 0.0090 | Val ROC-AUC 0.8510


Epoch 038 | Loss 0.0051 | Val ROC-AUC 0.8513


Epoch 039 | Loss 0.0067 | Val ROC-AUC 0.8515


Epoch 040 | Loss 0.0060 | Val ROC-AUC 0.8523
Early stopping at epoch 40.

FINAL | Test ROC-AUC: 0.9045 | Best epoch: 20


In [ ]:


import os
import csv
import random
import time
import numpy as np
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.datasets import MoleculeNet
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ==========================================
# 1. Dataset Loader & Splitter
# ==========================================
class IndexedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        data = self.dataset[real_idx].clone()
        data.graph_idx = torch.tensor([real_idx], dtype=torch.long)
        return data

def generate_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

def scaffold_split(dataset, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    random.seed(seed)

    scaffolds = defaultdict(list)

    # Build scaffold groups
    for i in range(len(dataset)):
        try:
            smiles = dataset[i].smiles
        except:
            smiles = dataset._data.smiles[i]

        scaffold = generate_scaffold(smiles)
        scaffolds[scaffold].append(i)

    # Shuffle scaffold groups
    scaffold_sets = list(scaffolds.values())
    random.shuffle(scaffold_sets)

    train_idx, val_idx, test_idx = [], [], []

    n_total = len(dataset)
    train_target = int(train_ratio * n_total)
    val_target   = int(val_ratio * n_total)

    # Greedy assignment
    for scaffold_set in scaffold_sets:
        if len(train_idx) + len(scaffold_set) <= train_target:
            train_idx.extend(scaffold_set)

        elif len(val_idx) + len(scaffold_set) <= val_target:
            val_idx.extend(scaffold_set)

        else:
            test_idx.extend(scaffold_set)

    # ------------------------------------------
    # Balance sanity check
    # ------------------------------------------
    def get_labels(indices):
        ys = []
        for idx in indices:
            y = dataset[idx].y.item()
            if not np.isnan(y):
                ys.append(int(y))
        return ys

    train_y = get_labels(train_idx)
    val_y   = get_labels(val_idx)
    test_y  = get_labels(test_idx)

    print("\nScaffold Split Statistics")
    print(f"Train: {len(train_idx)} | Pos {sum(train_y)} | Neg {len(train_y)-sum(train_y)}")
    print(f"Val:   {len(val_idx)} | Pos {sum(val_y)} | Neg {len(val_y)-sum(val_y)}")
    print(f"Test:  {len(test_idx)} | Pos {sum(test_y)} | Neg {len(test_y)-sum(test_y)}")

    # Ensure all splits contain both classes
    assert len(set(train_y)) > 1, "Train split has only one class"
    assert len(set(val_y)) > 1, "Validation split has only one class"
    assert len(set(test_y)) > 1, "Test split has only one class"

    return train_idx, val_idx, test_idx

def load_bbbp(batch_size=32, seed=42):
    dataset = MoleculeNet(root='./data', name='BBBP')

    # BBBP has ~2039 graphs. Let's do a simple 80/10/10 random split
    train_idx, val_idx, test_idx = scaffold_split(dataset, seed=42)
    train_dataset = IndexedDataset(dataset, train_idx)
    val_dataset   = IndexedDataset(dataset, val_idx)
    test_dataset  = IndexedDataset(dataset, test_idx)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
    return train_loader, val_loader, test_loader, dataset


def precompute_spectral_cache(dataset, desc="Pre-computing spectral bases"):
    cache = [None] * len(dataset)
    for i in tqdm(range(len(dataset)), desc=desc):
        data = dataset[i]
        n   = data.num_nodes
        ei  = data.edge_index

        adj = torch.zeros(n, n)
        adj[ei[0], ei[1]] = 1.0
        adj.fill_diagonal_(1.0)

        deg          = adj.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
        D_inv_sqrt   = torch.diag(deg_inv_sqrt)
        A_norm       = D_inv_sqrt @ adj @ D_inv_sqrt

        L = torch.eye(n) - A_norm
        try:
            _, U = torch.linalg.eigh(L)
            max_abs_idx = torch.abs(U).argmax(dim=0)
            signs       = torch.sign(U[max_abs_idx, torch.arange(n)])
            signs[signs == 0] = 1.0
            U = U * signs.unsqueeze(0)
        except Exception:
            U = torch.eye(n)

        cache[i] = (A_norm, U)

    total_bytes = sum(a.numel() * 4 + u.numel() * 4 for a, u in cache)
    print(f"\nCache built: {len(cache)} graphs | Size: {total_bytes / 1024**2:.1f} MB")
    return cache


class AtomEncoder(nn.Module):
    VOCAB_SIZES = [128, 8, 16, 16, 16, 8, 16, 4, 4]
    def __init__(self, hidden_dim):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(v, hidden_dim) for v in self.VOCAB_SIZES])
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = x.long()
        out = sum(emb(x[..., i].clamp(0, v - 1)) for i, (emb, v) in enumerate(zip(self.embeddings, self.VOCAB_SIZES)))
        return self.proj(F.gelu(out))

class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=3):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))

class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))

class GatedPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(), nn.Linear(hidden_dim // 2, 1))
    def forward(self, x, mask):
        scores  = self.gate(x).squeeze(-1).masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)

class GraphFNet_BBBP(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, out_dim=1, num_heads=4, lap_k=8, dropout=0.1, edge_dim=3):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)
        self.input_proj = AtomEncoder(hidden_dim)
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "local":  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                "global": SpectralMixMH(hidden_dim, num_heads=num_heads),
                "gate":   nn.Linear(hidden_dim, hidden_dim),
                "norm":   nn.LayerNorm(hidden_dim),
            }) for _ in range(num_layers)
        ])

        self.pool = GatedPooling(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, data, cache=None):
        x, mask = to_dense_batch(data.x.float(), data.batch)
        B, N, _ = x.shape

        A_list, U_list = [], []
        for b, gidx in enumerate(data.graph_idx.tolist()):
            A_b, U_b = cache[gidx]
            n = A_b.size(0)
            A_list.append(F.pad(A_b, (0, N-n, 0, N-n)).to(x.device))
            U_list.append(F.pad(U_b, (0, N-n, 0, N-n)).to(x.device))
        A_norm = torch.stack(A_list)
        U      = torch.stack(U_list)

        if data.edge_attr is not None:
            edge_attr_dense = to_dense_adj(data.edge_index, data.batch, edge_attr=data.edge_attr[:, :3].float(), max_num_nodes=N)
        else:
            edge_attr_dense = None

        x      = self.input_proj(x)
        k      = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        x      = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            x_res    = x
            x_local  = layer["local"](x, A_norm, edge_attr_dense)
            x_global = layer["global"](x, U, mask)
            gate     = torch.sigmoid(layer["gate"](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer["norm"](x_res + self.dropout(x_mix))

        x = x * mask.unsqueeze(-1)
        return self.classifier(self.pool(x, mask))

def evaluate_rocauc(model, loader, cache):
    model.eval()
    y_true_list, y_pred_list = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out   = model(batch, cache=cache)
            y_true_list.append(batch.y.cpu())
            y_pred_list.append(out.cpu())

    y_true = torch.cat(y_true_list, dim=0).numpy().flatten()
    y_pred = torch.cat(y_pred_list, dim=0).numpy().flatten()

    # Filter out NaNs (BBBP sometimes has missing labels depending on the PyG version)
    valid_idx = ~np.isnan(y_true)
    return roc_auc_score(y_true[valid_idx], y_pred[valid_idx])


def train_bbbp(seed=42, max_epochs=100, patience=20, batch_size=32):
    # Set seed for reproducible split
    set_seed(seed)

    print(f"\n========== SEED {seed} ==========")

    train_loader, val_loader, test_loader, full_dataset = load_bbbp(batch_size=batch_size,seed=seed)
    cache = precompute_spectral_cache(full_dataset)

    model = GraphFNet_BBBP().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_val, best_epoch, epochs_no_improve = 0.0, 0, 0
    ckpt_path = f"best_graphfnet_bbbp_seed{seed}.pt"

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch, cache=cache)
            valid_idx = ~torch.isnan(batch.y)

            loss = criterion(out[valid_idx], batch.y[valid_idx].float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        val_roc = evaluate_rocauc(model, val_loader, cache)
        improved = val_roc > best_val

        print(f"Epoch {epoch:03d} | Loss {total_loss/len(train_loader):.4f} | Val ROC-AUC {val_roc:.4f}" + (" ← best" if improved else ""))

        if improved:
            best_val, best_epoch, epochs_no_improve = val_roc, epoch, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(torch.load(ckpt_path, weights_only=False))
    test_roc = evaluate_rocauc(model, test_loader, cache)
    print(f"\nFINAL | Test ROC-AUC: {test_roc:.4f} | Best epoch: {best_epoch}")
    return test_roc

if __name__ == "__main__":

    print("Running 3-seed scaffold split evaluation")

    seeds = [0, 1, 2]
    results = []

    for seed in seeds:
        roc = train_bbbp(seed=seed)
        results.append(roc)

    results = np.array(results)

    print("\n===================================")
    print("FINAL 3-SEED RESULTS")
    print("===================================")

    for s, r in zip(seeds, results):
        print(f"Seed {s}: {r:.4f}")

    print(f"\nMean ROC-AUC: {results.mean():.4f}")
    print(f"Std  ROC-AUC: {results.std():.4f}")

Device: cuda
Running 3-seed scaffold split evaluation

========== SEED 0 ==========


[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:32] WARNING: not removing hydrogen atom without neighbors
[13:36:33] WARNING: not removing hydrogen atom without neighbors
[13:36:33] WARNING: not removing hydrogen atom without neighbors
[13:36:33] WARNING: not removing hydrogen atom without neighbors
[13:36:33] WARNING: not removing hydrogen atom without neighbors
[13:36:33] WARNING: not removing hydrogen atom without neighbors
[13:36:33] WARNING: not r


Scaffold Split Statistics
Train: 1631 | Pos 1233 | Neg 398
Val:   203 | Pos 167 | Neg 36
Test:  205 | Pos 160 | Neg 45
Train: 1631 | Val: 203 | Test: 205


Pre-computing spectral bases: 100%|██████████| 2039/2039 [00:01<00:00, 1648.58it/s]



Cache built: 2039 graphs | Size: 10.8 MB


...

Epoch 026 | Loss 0.0249 | Val ROC-AUC 0.8257


Epoch 027 | Loss 0.0171 | Val ROC-AUC 0.8275


Epoch 028 | Loss 0.0201 | Val ROC-AUC 0.8272


Epoch 029 | Loss 0.0151 | Val ROC-AUC 0.8298


Epoch 030 | Loss 0.0122 | Val ROC-AUC 0.8313
Early stopping at epoch 30.

FINAL | Test ROC-AUC: 0.8593 | Best epoch: 10

FINAL 3-SEED RESULTS
Seed 0: 0.8497
Seed 1: 0.8968
Seed 2: 0.8593

Mean ROC-AUC: 0.8686
Std  ROC-AUC: 0.0203
